# Predicting Angular Errors
<span style="font-family: 'Helvetica'; font-size: 15px;">
    Angular errors is nearly deterministic. In previous tests, we found that (______NEED NUMBER!_________)~90% of the variance in rx_azimuth_error_deg and rx_elevation_error_deg can be explained solely due to quantization. given by the equations:
</span>

$$ u 
= cos(el)sin(az)
$$

$$
v = sin(el)
$$


<span style="font-family: 'Helvetica'; font-size: 15px;">
    This happens because in ISAC simulations, the antenna can only "detect" a quantized number of degrees in the azimuth and elevation angles. We can recreate much of the error in the angle detection by computing the closest grid that the simulation will snap to. 
</span>


<span style="font-family: 'Helvetica'; font-size: 15px;">
    However, in roughly (NEED NUMBER) 1.5% of cases, detection seems to "slip". The measured angle is not the nearest grid, but the next grid beyond that. This seems to happens when (__________NEED EXPLANATION!________________). To account for this, we use a logistic regression model to predict whether a target will slip. Our tests (__________NEED EXPLANATION!______________) have also shown that the slipping mechanism for both the u and v quantities are very similar, so we choose to fit a single model for both to have more outlier samples in the training data.
</span>

<span style="font-family: 'Helvetica'; font-size: 15px;">
    To sample slippage, we use the SlipModel class. The training sequence is largely identical to the detection classifiers. Note that we can use the print_params argument to see the parameters of the logistic regression model. The features used are edge, close, rss, which are derived from rss_max, nn_norm_range_sep, and abs_frac_u/v. abs_frac_u/v is the absolute value of the fraction away from the nearest grid cell. Edge is the log distance to the next grid boundary, with a minimum distance added to prevent log(0). Close applies a gaussian radial basis on the distance to the neighbor. Very close targets in (u,v) receive a score near 1, while those far away get 0. For targets with null nn_norm_range_sep, in 1d target detection, they receive a score equal to the max nn_norm_range_sep in the other scenarios
</span>

In [ ]:
import sys
import polars as pl
import numpy as np
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.components.angular.slip import SlipModel
from src.data.load import load_pos, group_split

tags, labels, features = load_pos()
[X, X_val, y, y_val] = group_split(tags, labels, features)
model = SlipModel()
model.fit(X, y, print_params=True)

<span style="font-family: 'Helvetica'; font-size: 15px;">
    To sample the slip values, pass in the probabilities predicted by the logistic regression model, and a random number generator, along with your original inference data. This is because frac_u and frac_v are needed to predict the direction to snap towards. In some cases, the ISAC simulator snaps to the grid cell that is further away, at probabilitty ~6.5%. Additionally, ISAC sometimes slips more than 1 single cell away, so when we have a slip, we sample a magnitude from the distribution in the training set.
</span>

In [ ]:
u, v = model.predict_proba(X_val)
rng = np.random.default_rng(seed = 42)
samples = model.sample(X_val, u, v, rng)


<span style="font-family: 'Helvetica'; font-size: 15px;">
    SlipModel is only for testing and evaluation purposes. The AngleModel class abstracts away slip sampling and samples the full azimuth and elevation errors by calling a member SlipModel that it trains with fit(). Pass in the u, v probs into sample, and select az_err and el_err. Other columns like u_error and v_error are also included for further testing
</span>

In [ ]:
from src.components.angular.quantization import AngleModel

angle_model = AngleModel()
angle_model.fit(X,y)

u,v = angle_model.predict_proba(X_val)
samples = angle_model.sample(X_val, u, v, rng)
print(samples)

# 3D Positional Error

<span style="font-family: 'Helvetica'; font-size: 15px;">
    With the azimuth, range, and elevation errors, we have all the information needed to compute the full 3d positional error. The computation details are in src/geometry/jacobian.py. The FullPositionModel contains the complete pipeline to sample 3d position error. It fits member AngleModel and a member RangeModel using the training data (validation highly recommended). No manual probability prediction is offered, testing separate components should use the individual models.
</span>

In [ ]:
from src.components.position import FullPositionModel

pos_model = FullPositionModel()
tags, labels, features = load_pos()
[X, X_val, y, y_val, t, t_val] = group_split(tags, labels, features, return_tags=True)

train_df = pl.concat([X, y, t], how = "horizontal")
val_df = pl.concat([X_val, y_val, t_val], how = "horizontal")

pos_model.fit(train_df, val_df)

In [ ]:
pos_model.save("models/position_model.joblib")
pos_model = FullPositionModel.load("models/position_model.joblib")
pos_model.sample(X_val, rng)